<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_34_django_intro/note_lesson_34_django.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Урок 34 — Django: модель, ORM, view і адмінка

За два попередні уроки ми написали два сервери, які віддавали **дані** іншим програмам. Сьогодні — **сайт для людей**: сторінки з HTML, база даних, адмін-панель. Для цього є **Django** — вебфреймворк «з батарейками»: ORM, міграції, шаблони, адмінка, авторизація. З цього уроку росте застосунок нотаток, який дійде до повного Notes Chat App.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | Django-проєкт `hello_project` у ноутбуці | 0.1 | |
| 1 | проєкт, застосунок, `settings.py` | 1.1–1.3 | |
| 2 | MVT: маршрут → view → відповідь | 2.1–2.5 | |
| 3 | модель і міграції | 3.1–3.4 | |
| 4 | створення записів | 4.1–4.2 | |
| 5 | читання: `filter`, `exclude`, `order_by`, lookups | 5.1–5.4 | **Задача 1** |
| 6 | `get` і його винятки | 6.1–6.3 | |
| 7 | QuerySet лінивий; SQL під капотом | 7.1–7.4 | |
| 8 | зміна і видалення | 8.1–8.3 | |
| 9 | шаблони і view зі сторінкою | 9.1–9.3 | **Задача 2** |
| 10 | адмін-панель і паролі | 10.1–10.4 | |
| 11 | знайди помилку | 11.1 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**: приклади змінюють дані в базі, і наступні на це розраховують. Клітинка з визначенням (`def`, класи) лише створює функцію — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Інтернет не потрібен: Django і база SQLite працюють у ядрі ноутбука (у Colab інтернет потрібен лише першій клітинці).
- Теорія — у книзі курсу: [Урок 34](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_34/); поглиблено — у [Django-книзі викладача](https://nikoriakviktor.github.io/notes_chat_app/) (кроки 1–2 маршруту Zero to Hero).

---
## 0. Проєкт у ноутбуці

Готовий проєкт уроку — папка `hello_project` поруч із ноутбуком (модель `Note` з полями `title`, `content`, `is_pinned`, `priority`, `created_at`). У Colab ноутбук відкривається без файлів репозиторію — клітинка нижче завантажить лише папку уроку (`git clone --sparse`), встановить Django і застосує міграції (`manage.py migrate`) — створить базу `db.sqlite3`.

In [ ]:
import importlib.util
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_34_django_intro"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.exists("manage.py"):                # повторний запуск: ми вже в папці проєкту
    if not os.path.isdir("hello_project"):
        if not os.path.isdir("course_repo"):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
            subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
        os.chdir(os.path.join("course_repo", LESSON))
    os.chdir("hello_project")

if importlib.util.find_spec("django") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "Django>=5.2,<6"], check=True)

result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-3:]))

### Приклад 0.1. `django.setup()` — Django у звичайному Python

Зазвичай з Django працюють у терміналі (`python manage.py shell`, `runserver`). У ноутбуці робимо те саме з Python: вказуємо модуль налаштувань і викликаємо `django.setup()` — він завантажує налаштування й моделі.

In [ ]:
import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
# Jupyter виконує клітинки всередині asyncio-циклу, а ORM Django синхронний і за замовчуванням
# це забороняє. Для ноутбука дозволяємо явно (так радить документація Django).
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from django.contrib.auth.models import User
from hello_app.models import Note

Note.objects.all().delete()            # кожен запуск ноутбука — з порожньої таблиці
print("Django", django.get_version(), "| нотаток:", Note.objects.count())

---
## 1. Проєкт і застосунок

Django-сайт складається з **проєкту** — налаштувань усього сайту — і **застосунків** (apps): окремих частин за змістом. Проєкт створює `django-admin startproject hello_project .`, застосунок — `python manage.py startapp hello_app`.

### Приклад 1.1. Що лежить у папці проєкту

In [ ]:
from pathlib import Path

for path in sorted(Path(".").rglob("*")):
    if path.suffix in {".py", ".html"} and "__pycache__" not in path.parts:
        print(path)

| Файл | Навіщо |
|---|---|
| `manage.py` | командний рядок проєкту: сервер, міграції, shell |
| `hello_project/settings.py` | налаштування: застосунки, база даних, мова, часовий пояс |
| `hello_project/urls.py` | головна таблиця маршрутів |
| `hello_app/models.py`, `views.py`, `urls.py`, `admin.py` | моделі, view-функції, маршрути застосунку, адмінка |
| `hello_app/migrations/` | історія змін структури бази |
| `hello_app/templates/hello_app/` | HTML-шаблони застосунку |

### Приклад 1.2. Налаштування — звичайні змінні Python

In [ ]:
from django.conf import settings

print("INSTALLED_APPS:", settings.INSTALLED_APPS)
print("база:", settings.DATABASES["default"]["ENGINE"], "→", Path(settings.DATABASES["default"]["NAME"]).name)
print("мова:", settings.LANGUAGE_CODE, "| часовий пояс:", settings.TIME_ZONE, "| USE_TZ:", settings.USE_TZ)
print("DEBUG:", settings.DEBUG, "| ROOT_URLCONF:", settings.ROOT_URLCONF)

`hello_app` стоїть в `INSTALLED_APPS` — інакше Django не бачить його моделей, шаблонів і міграцій. Базу визначає налаштування, а не код: заміна SQLite на PostgreSQL — це зміна `DATABASES` (урок 39).

### Приклад 1.3. Реєстр застосунків

In [ ]:
from django.apps import apps

config = apps.get_app_config("hello_app")
print(config.name, "|", config.label, "|", Path(config.path).name)
print("моделі:", [model.__name__ for model in config.get_models()])
print("таблиця Note:", Note._meta.db_table)

Таблиця моделі називається `<застосунок>_<модель>` малими літерами: `hello_app_note`.

---
## 2. MVT: як Django відповідає на запит

```text
браузер ── GET /notes/ ──► urls.py (проєкт) ── include ──► hello_app/urls.py
                                                             │ path("notes/", views.note_list)
                                                             ▼
                                         view note_list(request)
                                           │ Note.objects.all()  ── SQL ──► db.sqlite3
                                           │ render(шаблон, {"notes": …})
                                           ▼
браузер ◄── 200 OK + HTML ─────────────── HttpResponse
```

**M**odel — дані, **V**iew — логіка, **T**emplate — HTML. Хто вирішує, яку view викликати, — таблиця маршрутів **URLconf**.

### Приклад 2.1. Яка view відповідає на адресу: `resolve`

In [ ]:
from django.urls import Resolver404, resolve, reverse


def show_route(path):
    try:
        match = resolve(path)
    except Resolver404:
        return f"{path:22} → маршруту немає (404)"
    return f"{path:22} → {match.func.__module__}.{match.func.__name__}  (ім'я: {match.view_name})"

In [ ]:
for path in ["/", "/about/", "/notes/", "/admin/", "/weather/"]:
    print(show_route(path))

`hello_app:note_list` — **ім'я** маршруту: `app_name = "hello_app"` у `hello_app/urls.py` плюс `name="note_list"` у `path(...)`.

### Приклад 2.2. Адреса за іменем: `reverse`

In [ ]:
for name in ["hello_app:index", "hello_app:about", "hello_app:note_list", "admin:index"]:
    print(f"{name:20} → {reverse(name)}")

Посилатися на маршрут за іменем (`reverse`, у шаблоні — `{% url "hello_app:note_list" %}`) краще, ніж писати `/notes/`: якщо адреса зміниться, правити доведеться лише `urls.py`.

### Приклад 2.3. Тестовий клієнт: запит без сервера

Сервер `runserver` у ноутбуці не запускаємо: Django має **тестовий клієнт**, який передає запит у view без мережі — так само, як браузер через `runserver`. `setup_test_environment()` дозволяє хост `testserver` і запам'ятовує, які шаблони використано.

In [ ]:
from django.test import Client, RequestFactory
from django.test.utils import setup_test_environment

setup_test_environment()
client = Client()

In [ ]:
for path in ["/", "/about/", "/weather/"]:
    response = client.get(path)
    print(path, response.status_code, response["Content-Type"], "|", response.content.decode()[:40])

На `/weather/` маршруту немає — Django сам відповідає `404` (поки `DEBUG = True`, сторінка помилки ще й перелічує маршрути — тому тіло довге). Рядок `Not Found: /weather/` над результатом — запис журналу Django (логер `django.request`), так само він з'являється в терміналі `runserver`.

### Приклад 2.4. View — звичайна функція

`RequestFactory` створює об'єкт запиту, і view можна викликати напряму, як будь-яку функцію: `request → HttpResponse`.

In [ ]:
from hello_app import views

factory = RequestFactory()
response = views.index(factory.get("/"))
print(type(response).__name__, response.status_code, response.content.decode())

### Приклад 2.5. Своя view: параметр із запиту

`request.GET` — параметри після `?`, як `params=` у `requests` (урок 32).

In [ ]:
from django.http import HttpResponse


def greet(request):
    name = request.GET.get("name", "світ")
    return HttpResponse(f"Привіт, {name}!")

In [ ]:
print(greet(factory.get("/greet/")).content.decode())
print(greet(factory.get("/greet/", {"name": "Олена"})).content.decode())

Щоб view стала сторінкою сайту, її додають у `urlpatterns`: `path("greet/", views.greet, name="greet")`.

---
## 3. Модель і міграції

**Модель** — Python-клас, що описує таблицю: атрибут-поле — колонка, екземпляр — рядок.

### Приклад 3.1. Поля моделі `Note`

In [ ]:
for field in Note._meta.get_fields():
    extra = []
    if getattr(field, "max_length", None):
        extra.append(f"max_length={field.max_length}")
    if field.has_default():
        extra.append(f"default={field.default!r}")
    if field.choices:
        extra.append(f"choices={len(field.choices)}")
    print(f"{field.name:11} {type(field).__name__:27} {', '.join(extra)}")
print("ordering:", Note._meta.ordering)

`id` Django додає сам — первинний ключ. `Meta.ordering = ["-is_pinned", "-created_at"]` — порядок за замовчуванням: мінус означає «за спаданням».

### Приклад 3.2. Який SQL виконує міграція

`makemigrations` порівнює моделі з історією і пише **файл міграції**; `migrate` виконує його на базі. `sqlmigrate` показує SQL — той самий `CREATE TABLE`, що в уроці 30. `call_command` запускає команду `manage.py` з Python.

In [ ]:
from io import StringIO

from django.core.management import call_command


def manage(*args):
    """Виконати команду manage.py і повернути її вивід рядком."""
    out = StringIO()
    call_command(*args, stdout=out)
    return out.getvalue()

In [ ]:
print(manage("sqlmigrate", "hello_app", "0001"))
print(manage("sqlmigrate", "hello_app", "0003"))

Міграція `0003` додала колонку `priority`. SQLite не вміє змінювати таблицю на місці, тому Django створює нову таблицю, копіює дані й перейменовує — нам про це думати не треба.

### Приклад 3.3. Які міграції застосовано

In [ ]:
print(manage("showmigrations", "hello_app"))

`[X]` — застосовано. Django пам'ятає це у службовій таблиці `django_migrations`, тому повторний `migrate` нічого не робить. Файли міграцій комітять у git: з ними база кожного розробника має ту саму структуру.

**🔮 Прогноз:** Навіщо полю `is_pinned` `default=False`, якщо його додали в модель, коли в таблиці вже були нотатки?

<details>
<summary>Відповідь</summary>

Міграція має знати, що записати в нову колонку для **наявних** рядків. Без `default` `makemigrations` спитав би значення інтерактивно.

</details>

### Приклад 3.4. Об'єкт без `save()` — лише в пам'яті

In [ ]:
draft = Note(title="Чернетка")
print("до save:", draft.pk, "| у базі:", Note.objects.count())
draft.save()
print("після save:", draft.pk, "| у базі:", Note.objects.count(), "|", draft.created_at is not None)
draft.delete()
print("після delete:", draft.pk, "| у базі:", Note.objects.count())

`Note(...)` існує лише в Python — `pk` ще `None`. `save()` робить `INSERT`, база дає `id`, а `auto_now_add` записує час створення. `delete()` прибирає рядок.

---
## 4. Створення записів

`Note.objects` — **менеджер**: вхідна точка до таблиці. `create(...)` = створити об'єкт і одразу `save()`.

### Приклад 4.1. Три нотатки

| title | content | is_pinned |
|---|---|---|
| `Купити молоко` | `2 л` | ні |
| `Django ORM` | `filter, get` | так |
| `Django admin` | *(порожньо)* | ні |

In [ ]:
Note.objects.create(title="Купити молоко", content="2 л")
Note.objects.create(title="Django ORM", content="filter, get", is_pinned=True)
Note.objects.create(title="Django admin")

print(Note.objects.count())
print([str(note) for note in Note.objects.all()])

**🔮 Прогноз:** У якому порядку `Note.objects.all()` повернув нотатки і чому «Django ORM» перша?

<details>
<summary>Відповідь</summary>

`Meta.ordering = ["-is_pinned", "-created_at"]`: спершу закріплені, потім від найновішої. «Django ORM» закріплена, далі «Django admin» (створена останньою), потім «Купити молоко».

</details>

### Приклад 4.2. Варіанти вибору: `priority`

In [ ]:
note = Note.objects.get(title="Купити молоко")
print(note.priority, "→", note.get_priority_display())
print(Note.PRIORITY_CHOICES)

У базі лежить число (`2`), а `get_priority_display()` повертає назву для людей. Для кожного поля з `choices` Django сам створює такий метод.

---
## 5. Читання: QuerySet

`filter` повертає **QuerySet** — набір записів, з яким можна працювати далі. Умови пишуть як `поле__умова=значення`:

| ORM | SQL |
|---|---|
| `title="Купити молоко"` | `title = 'Купити молоко'` |
| `title__icontains="django"` | `title LIKE '%django%'` без урахування регістру |
| `title__startswith="Ку"` | `title LIKE 'Ку%'` |
| `priority__gte=2` | `priority >= 2` |
| `id__in=[1, 3]` | `id IN (1, 3)` |

### Приклад 5.1. `filter` + `order_by` + `values_list`

In [ ]:
django_titles = list(Note.objects.filter(title__icontains="django").order_by("title").values_list("title", flat=True))
print(django_titles)

`values_list("title", flat=True)` — лише одна колонка, без створення об'єктів `Note`.

**🔮 Прогноз:** Чому `"Django ORM"` раніше за `"Django admin"` в алфавітному порядку?

<details>
<summary>Відповідь</summary>

SQLite порівнює рядки за кодами символів: великі латинські літери (`O` = 79) йдуть раніше за малі (`a` = 97). Порядок «як у словнику» залежить від collation бази; у PostgreSQL з українською локаллю він буде іншим.

</details>

### Приклад 5.2. `exclude` і `count`

In [ ]:
print("без тексту:", Note.objects.filter(content="").count())
print("з текстом:", [note.title for note in Note.objects.exclude(content="")])

`exclude` — «усі, крім»: `NOT (content = '')`. Порядок — з `Meta.ordering`: закріплена «Django ORM» перша.

### Приклад 5.3. Ще lookups — і пастка SQLite з кирилицею

In [ ]:
print("startswith 'Ку':", list(Note.objects.filter(title__startswith="Ку").values_list("title", flat=True)))
ids = list(Note.objects.order_by("id").values_list("id", flat=True))
print("id__in перших двох:", list(Note.objects.filter(id__in=ids[:2]).order_by("id").values_list("title", flat=True)))
print("priority__gte=2:", Note.objects.filter(priority__gte=2).count())
print("icontains 'купити':", Note.objects.filter(title__icontains="купити").count())
print("icontains 'Купити':", Note.objects.filter(title__icontains="Купити").count())

`icontains="купити"` не знайшов «Купити молоко»: у SQLite «без урахування регістру» працює лише для **латинських** літер. PostgreSQL порівнює без регістру будь-які літери — ще одна причина брати його в production (урок 39).

### Приклад 5.4. `values()` — словники замість об'єктів

In [ ]:
for row in Note.objects.values("title", "is_pinned", "priority"):
    print(row)

### Задача 1. `search_titles(text, pinned=None)`

Напиши функцію, яка повертає **список** заголовків нотаток, у заголовку яких є `text` (без урахування регістру), у порядку за замовчуванням (`Meta.ordering`).

- якщо `pinned` — `True` чи `False`, залиш лише закріплені чи лише незакріплені;
- якщо `pinned` — `None`, фільтра за закріпленням немає;
- один QuerySet, результат — `list`, а не QuerySet.

Перевірка — у наступній клітинці.

In [ ]:
def search_titles(text, pinned=None):
    # BEGIN SOLUTION
    notes = Note.objects.filter(title__icontains=text)
    if pinned is not None:
        notes = notes.filter(is_pinned=pinned)
    return list(notes.values_list("title", flat=True))
    # END SOLUTION

In [ ]:
result = search_titles("django")
assert isinstance(result, list), f"потрібен list, а не {type(result).__name__}"
assert result == ["Django ORM", "Django admin"], f"порядок за замовчуванням: закріплені першими; отримали {result}"
assert search_titles("DJANGO", pinned=True) == ["Django ORM"]
assert search_titles("django", pinned=False) == ["Django admin"]
assert search_titles("admin") == ["Django admin"]
assert search_titles("python") == []
print("✅ Задача 1 розв'язана")

Умову `pinned` ми додаємо **окремим** `filter` лише тоді, коли вона потрібна: QuerySet можна нарощувати крок за кроком, а SQL однаково буде один (розділ 7).

---
## 6. `get` і його винятки

`get()` повертає рівно **один** об'єкт, інакше — `Note.DoesNotExist` або `Note.MultipleObjectsReturned`.

### Приклад 6.1. Один, жодного, кілька

In [ ]:
note = Note.objects.get(title="Django ORM")
print(note.title, "|", note.content)

try:
    Note.objects.get(title="Немає такої")
except Note.DoesNotExist as error:
    print("DoesNotExist:", error)

try:
    Note.objects.get(title__icontains="django")
except Note.MultipleObjectsReturned as error:
    print("MultipleObjectsReturned:", error)

### Приклад 6.2. «Нотатка або `None`»

In [ ]:
def note_or_none(pk):
    try:
        return Note.objects.get(pk=pk)
    except Note.DoesNotExist:
        return None

In [ ]:
first = Note.objects.order_by("id").first()
print(note_or_none(first.pk), "|", note_or_none(10**6))
assert note_or_none(first.pk) == first and note_or_none(10**6) is None

`first()` теж повертає `None`, якщо записів немає, — але бере **перший** з багатьох, а не перевіряє, що він один.

### Приклад 6.3. У view — `get_object_or_404`

У view «не знайдено» має стати відповіддю `404`, а не помилкою сервера `500`. `get_object_or_404` робить `get` і кидає `Http404`, який Django перетворює на сторінку 404.

In [ ]:
from django.http import Http404
from django.shortcuts import get_object_or_404

print(get_object_or_404(Note, title="Django ORM"))
try:
    get_object_or_404(Note, pk=10**6)
except Http404 as error:
    print("Http404:", error)

---
## 7. QuerySet лінивий

QuerySet не йде в базу, доки не знадобляться дані: `for`, `list`, `len`, `print`, індекс, `first()`, `count()`, `exists()`. Тому ланцюжок `filter(...).exclude(...).order_by(...)` — це **один** SQL-запит.

### Приклад 7.1. Який SQL надішле Django: `.query`

In [ ]:
urgent = Note.objects.filter(is_pinned=True, priority__gte=2)
sql = str(urgent.query)
print(sql)
print(urgent.count())

`ORDER BY` з'явився сам — з `Meta.ordering`. `.query` показує SQL **без** виконання.

### Приклад 7.2. Рахуємо запити до бази

`CaptureQueriesContext` записує всі SQL-запити, виконані всередині блоку `with`.

In [ ]:
from django.db import connection
from django.test.utils import CaptureQueriesContext


def count_queries(fn):
    """Виконати fn() і повернути (результат, скільки SQL-запитів пішло в базу)."""
    with CaptureQueriesContext(connection) as captured:
        result = fn()
    return result, len(captured.captured_queries)

In [ ]:
chain, n = count_queries(lambda: Note.objects.filter(priority__gte=1).exclude(content="").order_by("title"))
print("після побудови ланцюжка:", n, "запитів")
titles, n = count_queries(lambda: [note.title for note in chain])
print("після циклу:", n, "запит —", titles)

### Приклад 7.3. Кеш QuerySet

In [ ]:
def twice_same():
    notes = Note.objects.all()
    return len(notes), len(notes)          # той самий QuerySet двічі


def twice_new():
    return len(Note.objects.all()), len(Note.objects.all())   # два нові QuerySet

In [ ]:
print("той самий QuerySet:", count_queries(twice_same))
print("два нові QuerySet: ", count_queries(twice_new))

Виконаний QuerySet запам'ятовує результат: другий `len` того самого об'єкта не йде в базу. Новий `Note.objects.all()` — новий запит.

### Приклад 7.4. `count()` проти `len()`

In [ ]:
with CaptureQueriesContext(connection) as captured:
    Note.objects.count()
    len(Note.objects.all())
for query in captured.captured_queries:
    print(query["sql"][:110])

`count()` просить у бази лише число (`SELECT COUNT(*)`), а `len()` завантажує **всі рядки** в Python і рахує їх там. Якщо потрібне лише число — `count()`; якщо дані однаково знадобляться — `len()` вже виконаного QuerySet.

---
## 8. Зміна і видалення

### Приклад 8.1. Один об'єкт: змінити атрибут і `save()`

In [ ]:
note = Note.objects.get(title="Купити молоко")
note.content = "2 л, і хліб"
note.save()
print(Note.objects.get(pk=note.pk).content)

### Приклад 8.2. Багато рядків одним SQL: `update()`

Закріпимо **всі** нотатки, в заголовку яких є `django`.

In [ ]:
updated = Note.objects.filter(title__icontains="django").update(is_pinned=True)
print("оновлено:", updated)
print("закріплених:", Note.objects.filter(is_pinned=True).count())

`update()` — один `UPDATE … WHERE …` без завантаження об'єктів у Python і повертає кількість рядків. `save()` — для одного вже завантаженого об'єкта.

### Приклад 8.3. `delete()` повертає, скільки видалено

In [ ]:
Note.objects.create(title="Тимчасова")
print(Note.objects.filter(title="Тимчасова").delete())
print(Note.objects.filter(title="Тимчасова").delete())
print(list(Note.objects.values_list("title", "is_pinned")))

`(1, {'hello_app.Note': 1})` — усього рядків і розбивка за моделями (коли є зв'язки, видаляються й залежні записи). Удруге видаляти нічого — `(0, {})`.

---
## 9. Шаблон: дані з бази на сторінці

View бере дані з моделі й передає їх у **шаблон** — HTML з мовою шаблонів Django:

- `{{ … }}` — вставити значення; `|length`, `|default:"…"` — **фільтри**;
- `{% for %} … {% empty %} … {% endfor %}` — цикл; `empty` — коли список порожній.

### Приклад 9.1. Шаблон з рядка: фільтри, цикл і автоекранування

In [ ]:
from django.template import Context, Template

template = Template(
    "Нотаток: {{ notes|length }}\n"
    "{% for note in notes %}- {{ note.title }} — {{ note.content|default:'без тексту' }}\n"
    "{% empty %}- нотаток ще немає\n{% endfor %}"
)

In [ ]:
print(template.render(Context({"notes": Note.objects.all()})))
print(template.render(Context({"notes": []})))
print(Template("{{ title }}").render(Context({"title": "<script>alert(1)</script>"})))

Django **екранує** HTML у значеннях: нотатка з заголовком `<script>` покажеться як текст `&lt;script&gt;`, а не виконається (захист від XSS — урок 41).

### Приклад 9.2. Шаблон застосунку: `render_to_string`

Шаблони Django шукає в папці `templates/` кожного застосунку; вкладена папка `hello_app/` захищає від конфлікту однакових імен.

In [ ]:
from django.template.loader import render_to_string

html = render_to_string("hello_app/note_list.html", {"notes": Note.objects.all()})
print(html)

### Приклад 9.3. Сторінка `/notes/` через тестовий клієнт

`views.note_list` робить те саме: `render(request, "hello_app/note_list.html", {"notes": notes})`.

In [ ]:
response = client.get("/notes/")
html = response.content.decode()
print(response.status_code, "| шаблони:", [t.name for t in response.templates])
print("у контексті:", list(response.context["notes"]))
print(html[html.index("<h1>"):html.index("</h1>") + 5])
assert html.index("Django ORM") < html.index("Купити молоко"), "закріплені — першими"

Тестовий клієнт показує не лише HTML, а й **які шаблони** використано і **що передано** в контекст — так перевіряють view у тестах (урок 42).

### Задача 2. View однієї нотатки

Напиши view `note_detail(request, pk)`:

- знаходить нотатку за `pk` через `get_object_or_404` (якщо немає — вилітає `Http404`);
- повертає `HttpResponse` з текстом `"<title>: <content>"`, а для порожнього тексту — `"<title>: без тексту"`.

Перевіряємо викликом з `RequestFactory`, як у прикладі 2.4. Перевірка — у наступній клітинці.

In [ ]:
def note_detail(request, pk):
    # BEGIN SOLUTION
    note = get_object_or_404(Note, pk=pk)
    return HttpResponse(f"{note.title}: {note.content or 'без тексту'}")
    # END SOLUTION

In [ ]:
milk = Note.objects.get(title="Купити молоко")
admin_note = Note.objects.get(title="Django admin")
response = note_detail(factory.get(f"/notes/{milk.pk}/"), milk.pk)
assert isinstance(response, HttpResponse), "view має повертати HttpResponse"
assert response.content.decode() == "Купити молоко: 2 л, і хліб", response.content.decode()
assert note_detail(factory.get("/"), admin_note.pk).content.decode() == "Django admin: без тексту"
try:
    note_detail(factory.get("/"), 10**6)
    raise AssertionError("для неіснуючої нотатки очікували Http404")
except Http404:
    pass
print(response.content.decode())
print("✅ Задача 2 розв'язана")

Щоб це стало сторінкою, у `hello_app/urls.py` додають `path("notes/<int:pk>/", views.note_detail, name="note_detail")`: `<int:pk>` — частина адреси, яка стане аргументом `pk`. Це — крок уроку 35.

---
## 10. Адмін-панель

Django **сам** будує адмінку для зареєстрованих моделей (`hello_app/admin.py`): список, пошук, фільтри, форми. Увійти може лише персонал (`is_staff`), усе бачить суперкористувач.

### Приклад 10.1. Без входу — перенаправлення на логін

In [ ]:
anonymous = client.get("/admin/hello_app/note/")
print(anonymous.status_code, "→", anonymous["Location"])

### Приклад 10.2. Суперкористувач і вхід

Зазвичай його створює `python manage.py createsuperuser`. Тут — з Python: створюємо (або оновлюємо) користувача `teacher` і входимо клієнтом.

In [ ]:
teacher, created = User.objects.get_or_create(username="teacher", defaults={"email": "teacher@example.com"})
teacher.is_staff = teacher.is_superuser = True
teacher.set_password("lesson-34")
teacher.save()
print("створено" if created else "вже був", "| вхід:", client.login(username="teacher", password="lesson-34"))

admin_page = client.get("/admin/hello_app/note/")
print(admin_page.status_code, "| «Купити молоко» на сторінці:", "Купити молоко" in admin_page.content.decode())

### Приклад 10.3. Пошук і фільтр адмінки — це параметри URL

`search_fields` дає рядок пошуку (`?q=`), `list_filter` — фільтр праворуч (`?is_pinned__exact=0`). Порахуємо нотатки на сторінці через контекст відповіді.

In [ ]:
def admin_titles(query=""):
    response = client.get("/admin/hello_app/note/" + query)
    return [note.title for note in response.context["cl"].result_list]

In [ ]:
print("усі:              ", admin_titles())
print("пошук 'admin':    ", admin_titles("?q=admin"))
print("незакріплені:     ", admin_titles("?is_pinned__exact=0"))

Адмінка — це звичайні view Django поверх того самого ORM: пошук — `icontains` по `title` і `content`, фільтр — `filter(is_pinned=False)`.

**🔮 Прогноз:** Що лежить у полі `User.objects.get(username="teacher").password` — сам пароль?

<details>
<summary>Відповідь</summary>

Ні: рядок `pbkdf2_sha256$<раунди>$<сіль>$<хеш>`. Django зберігає лише хеш, а `check_password` порівнює хеші (урок 17, урок 41).

</details>

### Приклад 10.4. Пароль — лише хеш

In [ ]:
stored = User.objects.get(username="teacher").password
algorithm, iterations, salt, digest = stored.split("$")
print(algorithm, "| раундів:", iterations, "| сіль:", len(salt), "символів")
print(teacher.check_password("lesson-34"), teacher.check_password("qwerty"))

Сотні тисяч раундів хешування із сіллю роблять підбір пароля дорогим. Докладно — в уроці 41.

---
## 11. Знайди помилку

### Приклад 11.1. `get` там, де записів може бути кілька

In [ ]:
def first_unpinned_title_buggy():
    return Note.objects.get(is_pinned=False).title

In [ ]:
print("сьогодні працює:", first_unpinned_title_buggy())
extra = Note.objects.create(title="Ще одна")
try:
    first_unpinned_title_buggy()
except Note.MultipleObjectsReturned as error:
    print("дві незакріплені:", type(error).__name__)
extra.delete()
print("прибрали зайву — знову працює:", first_unpinned_title_buggy())

Поки незакріплена нотатка одна, функція працює. Щойно їх дві — `MultipleObjectsReturned`, а коли не стане жодної — `DoesNotExist`. `get` — для пошуку **за унікальним** полем (`pk`), а «перша з кількох» — це `filter(...).order_by(...).first()`.

### Задача 3. Правильна версія

Перепиши `first_unpinned_title()`: повертає заголовок **найновішої** незакріпленої нотатки або `None`, якщо таких немає. Без `try/except`.

Перевірка — у наступній клітинці.

In [ ]:
def first_unpinned_title():
    # BEGIN SOLUTION
    note = Note.objects.filter(is_pinned=False).order_by("-created_at").first()
    return note.title if note else None
    # END SOLUTION

In [ ]:
assert first_unpinned_title() == "Купити молоко", "очікували «Купити молоко» — єдину незакріплену нотатку"
Note.objects.create(title="Ще одна")
assert first_unpinned_title() == "Ще одна", "з двох незакріплених — найновіша"
Note.objects.filter(is_pinned=False).update(is_pinned=True)
assert first_unpinned_title() is None, "немає незакріплених — None"
print("✅ Задача 3 розв'язана")

---
## Самоперевірка

1. Що робить `makemigrations`, а що — `migrate`?
2. Опиши шлях запиту `GET /notes/` у Django.
3. Чим `filter()` відрізняється від `get()`? Які винятки може кинути `get()`?
4. Коли QuerySet іде в базу? Чим `count()` відрізняється від `len()`?
5. Чим `note.save()` відрізняється від `QuerySet.update()`?
6. Навіщо `app_name` і `name` у маршрутах?

<details>
<summary>Відповіді</summary>

1. `makemigrations` створює файл міграції з опису моделей; `migrate` виконує міграції на базі й записує їх у `django_migrations`.
2. `urls.py` проєкту → `include` → `urls.py` застосунку → `note_list(request)` → `Note.objects.all()` → SQL до бази → `render` шаблону з контекстом → `HttpResponse` з HTML.
3. `filter()` — QuerySet з 0…N записів; `get()` — рівно один об'єкт або виняток `DoesNotExist` / `MultipleObjectsReturned`.
4. Коли потрібні дані: цикл, `list()`, `len()`, `print()`, індекс, `first()`, `count()`, `exists()`. `count()` — `SELECT COUNT(*)`, `len()` завантажує всі рядки.
5. `save()` зберігає один завантажений об'єкт; `update()` — один SQL `UPDATE` для всіх рядків QuerySet.
6. Щоб посилатися на маршрут за ім'ям (`reverse("hello_app:note_list")`, `{% url %}`), а не за жорстко прописаною адресою.

</details>

## Далі

- Книга курсу: [Урок 34](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_34/) — створення проєкту з нуля, міграції та їхній SQL, адмінка зі скріншотами.
- Django-книга: [крок 1](https://nikoriakviktor.github.io/notes_chat_app/tutorials/01_hello_django/) і [крок 2](https://nikoriakviktor.github.io/notes_chat_app/tutorials/02_first_model/) маршруту Zero to Hero.
- Запусти проєкт у терміналі: `python manage.py runserver` у папці `hello_project` і відкрий `/notes/` та `/admin/`.
- **Урок 35** — форми, base-шаблон, Bootstrap і CRUD нотаток.